In [ ]:
import io
import os
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from office365.sharepoint.client_context import ClientContext
from office365.sharepoint.files.file import File

load_dotenv()

USERNAME = os.getenv("SHAREPOINT_USERNAME")
PASSWORD = os.getenv("SHAREPOINT_PASSWORD")
TENANT = os.getenv("SHAREPOINT_TENANT")
CLIENT_ID = os.getenv("SHAREPOINT_CLIENT_ID")

SHAREPOINT_SITE = "https://dannythomaspartyrentals-my.sharepoint.com/personal/jose_dannythomaspartyrentals_com"
SHAREPOINT_SITE_NAME = "personal/jose_dannythomaspartyrentals_com"
SHAREPOINT_DOC = "Documents"
FOLDER_NAME = "Dispatch"
TARGET_SHEET = "Log"

ctx = ClientContext(SHAREPOINT_SITE).with_username_and_password(
    TENANT, CLIENT_ID, USERNAME, PASSWORD
)

target_folder_url = f"{SHAREPOINT_DOC}/{FOLDER_NAME}"
folder = ctx.web.get_folder_by_server_relative_url(target_folder_url)
folder.expand(["Files"]).get().execute_query()

excel_files = [f.properties["Name"] for f in folder.files
               if f.properties["Name"].endswith((".xlsx", ".xls", ".xlsm"))]

def parse_date(filename):
    name = filename.replace(".xlsm", "").split(" ")[0]
    try:
        return datetime.strptime(name[:6], "%m%d%y")
    except ValueError:
        return datetime.max

excel_files = sorted(excel_files, key=parse_date)
print(f"Excel files ({len(excel_files)}):")
for f in excel_files:
    print(f"  - {f}")

all_data = {}

for file_name in excel_files:
    file_url = f"/{SHAREPOINT_SITE_NAME}/{SHAREPOINT_DOC}/{FOLDER_NAME}/{file_name}"
    file_bytes = File.open_binary(ctx, file_url).content

    sheets = pd.read_excel(io.BytesIO(file_bytes), sheet_name=None, engine="openpyxl")
    log_sheet = next((name for name in sheets if name.strip().lower() == TARGET_SHEET.lower()), None)

    if log_sheet:
        all_data[file_name] = sheets[log_sheet]
        print(f"\nFile: {file_name}  |  Sheet: {log_sheet}  |  Shape: {sheets[log_sheet].shape}")
    else:
        print(f"\nFile: {file_name}  |  '{TARGET_SHEET}' sheet not found. Available: {list(sheets)}")

# Access any sheet: all_data["092326.xlsm"]